# Match the existing company-grievance inputs with `itk-match`

This notebook uses **`packages/itk-match`** (`import itk_match`), not the legacy Streamlit workflow. It reads these two repository inputs directly, without copying them or generating datasets:

- `example_outputs\match_entity_records\company_grievances\company_grievances_input_data_1.csv`
- `example_outputs\match_entity_records\company_grievances\company_grievances_input_data_2.csv`

Both have 2,000 rows, but their column names differ. We align their schemas and show observed matching groups, unmatched records, and threshold effects. **No verified identity labels are supplied.** Historical matching output files are not read or treated as authoritative ground truth, so group counts are not accuracy, precision, or recall measurements.

We use the real `Dataset`, `Matcher.match`, `MatchConfig`, and `Matcher.regroup` API. A custom character-trigram embedder makes execution deterministic and offline after dependency installation: no API key, hosted LLM, pretrained model download, or paid call. This is a **lexical baseline**, not a demonstration of neural embedding quality.

## 1. Install and select a Python kernel

Use **Python 3.11 or 3.12** (the packages require `>=3.11,<3.13`). From the repository root, run these commands once in a terminal:

```powershell
uv venv --python 3.12 .venv-match-demo
uv pip install --python .venv-match-demo\Scripts\python.exe -e .\packages\itk-core -e .\packages\itk-match nbformat nbclient ipykernel
.\.venv-match-demo\Scripts\python.exe -m ipykernel install --user --name itk-match-demo --display-name "Python (itk-match demo)"
```

On macOS/Linux use `.venv-match-demo/bin/python` and slash-separated paths instead. Install **both local packages together**: `itk-core` on PyPI is a different project, and these packages are bundled in the Intelligence Toolkit wheel rather than independently published.

Open this notebook in VS Code or Jupyter, select **Python (itk-match demo)**, and run all cells. Jupyter's top-level `await` runs the async API directly. Input lookup supports the repository root and this notebook's folder. Installed versions are displayed for the saved run. All 4,000 records are processed; displayed tables are explicitly labeled previews, and the complete results remain in `result`, `matched`, `unmatched`, and `candidate_details`. No new output CSVs are written.

In [1]:
# Copyright (c) 2026 Microsoft Corporation. All rights reserved.
# Licensed under the MIT license. See LICENSE file in the project.

import re
from collections import Counter
from importlib.metadata import version
from pathlib import Path
from typing import ClassVar

import numpy as np
import polars as pl
from IPython.display import Markdown
from IPython.display import display as show
from itk_match import Dataset, MatchConfig, Matcher
from itk_match.sentences import to_sentences

pl.Config.set_tbl_rows(25)
pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(45)

cwd = Path.cwd().resolve()
relative_inputs = (
    Path("example_outputs") / "match_entity_records" / "company_grievances"
)
input_names = [
    "company_grievances_input_data_1.csv",
    "company_grievances_input_data_2.csv",
]
repo_root = next(
    (
        parent
        for parent in (cwd, *cwd.parents)
        if all(
            (parent / relative_inputs / filename).is_file() for filename in input_names
        )
    ),
    None,
)
if repo_root is None:
    message = "Run from within the repository containing both company-grievance inputs."
    raise FileNotFoundError(message)
input_paths = [repo_root / relative_inputs / filename for filename in input_names]
for path in input_paths:
    print("Input:", path.relative_to(repo_root))
show(
    pl.DataFrame({
        "package": ["itk-match", "itk-core", "numpy", "polars", "rapidfuzz"],
        "version": [
            version(name)
            for name in ["itk-match", "itk-core", "numpy", "polars", "rapidfuzz"]
        ],
    })
)

Input: example_outputs\match_entity_records\company_grievances\company_grievances_input_data_1.csv
Input: example_outputs\match_entity_records\company_grievances\company_grievances_input_data_2.csv


package,version
str,str
"""itk-match""","""0.1.0"""
"""itk-core""","""0.1.0"""
"""numpy""","""2.5.4"""
"""polars""","""2.1.0"""
"""rapidfuzz""","""3.14.6"""


## 2. Inspect both sources and align their schemas

`Dataset.attributes` maps **source column -> canonical label**. Source 1 uses `employer_id`/`employer_name`; source 2 uses `company_id`/`company_name`. The seven remaining columns describe sector, address, city, country, email, phone, and owner. We align all seven; IDs identify records and are **not embedded**.

All CSV columns are read as strings to preserve identifiers and phone formatting. Dataset names namespace IDs into `data_1:<id>` and `data_2:<id>`; equal numeric IDs across sources do not imply a match. The table below checks missing/blank names and ID uniqueness, followed by input previews. Similar names, email domains, phone numbers, and owners can occur in several records, so none is a verified identity label.

In [2]:
data_1, data_2 = [pl.read_csv(path, infer_schema=False) for path in input_paths]
datasets = [
    Dataset(
        name="data_1",
        frame=data_1,
        name_column="employer_name",
        id_column="employer_id",
        attributes={
            "sector": "sector",
            "address": "address",
            "city": "city",
            "country": "country",
            "email": "email",
            "phone": "phone",
            "owner": "owner",
        },
    ),
    Dataset(
        name="data_2",
        frame=data_2,
        name_column="company_name",
        id_column="company_id",
        attributes={
            "industry_sector": "sector",
            "street_address": "address",
            "city_address": "city",
            "country_address": "country",
            "email_address": "email",
            "phone_number": "phone",
            "company_owner": "owner",
        },
    ),
]
input_summary = []
valid_input_rows = 0
for dataset in datasets:
    names = dataset.frame[dataset.name_column].to_list()
    valid_names = sum(name is not None and bool(name.strip()) for name in names)
    ids = dataset.frame[dataset.id_column]
    assert ids.null_count() == 0
    assert ids.n_unique() == dataset.frame.height
    assert all(value.strip() for value in ids.to_list())
    valid_input_rows += valid_names
    input_summary.append({
        "source": dataset.name,
        "rows": dataset.frame.height,
        "columns": dataset.frame.width,
        "valid_names": valid_names,
        "missing_or_blank_names": dataset.frame.height - valid_names,
        "unique_ids": ids.n_unique(),
    })
    show(Markdown(f"**{dataset.name}: first 10 of {dataset.frame.height} input rows**"))
    show(dataset.frame.head(10))
show(Markdown("**Input summary (all rows)**"))
show(pl.DataFrame(input_summary))

**data_1: first 10 of 2000 input rows**

employer_id,employer_name,sector,address,city,country,email,phone,owner
str,str,str,str,str,str,str,str,str
"""1""","""Adventure Awaits""","""Travel""","""1212 Journey Rd""","""Explore City""","""Traveland""","""info@adventureawaits.com""","""1111000999""","""Taylor Journey"""
"""2""","""Adventure Gear""","""Retail""","""1010 Trekking Trl""","""Adventure City""","""Exploreland""","""contact@adventuregear.retail""","""1555000115""","""Morgan Trek"""
"""3""","""Adventure Gear Co.""","""Retail""","""555 Adventure St""","""Outdoor City""","""ExploreLand""","""info@adventuregear.com""","""6655443322""","""Explorer Jane"""
"""4""","""Adventure Gear Company""","""Retail""","""555 Adventure Street""","""Outdoor City""","""ExploreLand""","""contact@adventuregear.com""","""6655443322""","""Explorer Jane"""
"""5""","""Adventure Outdoors""","""Recreation""","""654 Adventure Ln""","""Outdoorsville""","""Explorica""","""info@adventureoutdoors.com""","""3344556677""","""Alex Explorer"""
"""6""","""Adventure Seekers""","""Travel""","""404 Adventure Blvd""","""Explore City""","""Travelia""","""info@adventureseekers.com""","""1555666782""","""Lara Quest"""
"""7""","""Adventure Seekers Travel""","""Travel""","""369 Adventure Ave""","""Adventurist""","""Wanderland""","""contact@adventureseekers.com""","""5544332211""","""Explorer Max"""
"""8""","""Adventure Tours""","""Travel""","""1414 Journey Ave""","""Explore City""","""Traveland""","""info@adventuretours.com""","""3334445566""","""Ms. Adventure"""
"""9""","""Adventure Tours Co.""","""Travel""","""754 Adventure St""","""TravelCity""","""ExploreLand""","""contact@adventuretours.com""","""3344556678""","""Jamie Explore"""


**data_2: first 10 of 2000 input rows**

company_id,company_name,industry_sector,street_address,city_address,country_address,email_address,phone_number,company_owner
str,str,str,str,str,str,str,str,str
"""1""","""ActiveWear Ltd.""","""Sports""","""789 Workout Rd""","""Sportstown""","""Athletica""","""hello@activewear.com""","""9900112233""","""Taylor Exercise"""
"""2""","""Adventure Gear""","""Retail""","""963 Adventure Rd""","""GearCity""","""ExploreLand""","""info@adventuregear.com""","""1415161718""","""Charlotte Black"""
"""3""","""Adventure Heights""","""Tourism""","""511 Summit St""","""Highland""","""Mountland""","""contact@adventureheights.com""","""6677889904""","""Climber Hill"""
"""4""","""Adventure Outdoors""","""Recreation""","""404 Adventure Ln""","""Explore City""","""Adventuria""","""info@adventureoutdoors.com""","""1555666777""","""Trail Blazer"""
"""5""","""Adventure Outfitters""","""Retail""","""558 Adventure St""","""Outdoor City""","""ExploreLand""","""sales@adventureoutfitters.com""","""6655443325""","""Explorer Jane"""
"""6""","""Adventure Planners""","""Tourism""","""1010 Adventure Road""","""Travel City""","""Tourland""","""info@adventureplanners.adv""","""9988776656""","""Journey Master"""
"""7""","""Adventure Sports Co.""","""Sports""","""864 Adventure Road""","""Sport City""","""Playland""","""contact@adventuresports.com""","""1434455667""","""Morgan Sport"""
"""8""","""Adventure Supplies""","""Retail""","""560 Adventure St""","""Outdoor City""","""ExploreLand""","""info@adventuresupplies.com""","""6655443327""","""Explorer Ella"""
"""9""","""Adventure Tours""","""Travel""","""999 Adventure Rd""","""Travel City""","""WanderLand""","""contact@adventuretours.com""","""3344556677""","""Liam Cyan"""


**Input summary (all rows)**

source,rows,columns,valid_names,missing_or_blank_names,unique_ids
str,i64,i64,i64,i64,i64
"""data_1""",2000,9,2000,0,2000
"""data_2""",2000,9,2000,0,2000


## 3. A transparent offline embedder

`Matcher` creates `NAME: ...; ADDRESS: ...; CITY: ...; COUNTRY: ...; EMAIL: ...; OWNER: ...; PHONE: ...; SECTOR: ...` sentences and calls an object implementing `embed_store_many`. The package then performs cosine candidate blocking, name scoring, and union-find clustering itself.

Our adapter lowercases text, removes punctuation, normalizes a fixed list of equivalent abbreviations, and counts character trigrams over each whole sentence. A sorted corpus vocabulary yields deterministic aligned NumPy vectors without random seeds or hash collisions. **It never reads source IDs, identity labels, or historical output files.** Vocabulary is rebuilt per embedding call; it is not a persistent production vector store.

This English-language lexical baseline weights repeated/common trigrams through raw counts. Shared labels, sector terms, email components, or phone numbers may inflate similarity; conflicting locations may lower it. A small abbreviation map is not a general address parser. All rows are embedded jointly in a dense in-memory matrix.

In [3]:
class OfflineTrigramEmbedder:
    """Embed one joint sentence batch as deterministic character-trigram counts."""

    abbreviations: ClassVar[dict[str, str]] = {
        "street": "st",
        "road": "rd",
        "avenue": "ave",
        "boulevard": "blvd",
        "drive": "dr",
        "lane": "ln",
        "limited": "ltd",
        "incorporated": "inc",
    }

    def __init__(self):
        self.calls = 0

    def features(self, text):
        """Normalize common abbreviations and count sentence character trigrams."""
        words = re.findall(r"[a-z0-9]+", text.lower())
        normalized = " ".join(self.abbreviations.get(word, word) for word in words)
        return Counter(normalized[i : i + 3] for i in range(len(normalized) - 2))

    async def embed_store_many(self, data, callbacks=None, cache_data=True):
        """Return each input payload with its vector; no API or cache is used."""
        self.calls += 1
        features = [self.features(item["text"]) for item in data]
        vocabulary = sorted({gram for counts in features for gram in counts})
        return [
            {
                **item,
                "vector": np.array(
                    [counts[gram] for gram in vocabulary], dtype=np.float32
                ),
            }
            for item, counts in zip(data, features, strict=True)
        ]


embedder = OfflineTrigramEmbedder()
matcher = Matcher(embedder=embedder)

## 4. Configure and run matching

- **`max_distance=0.15`**: candidate cosine distance must be at most 0.15 (cosine similarity at least 0.85). Smaller is stricter. This is an illustrative setting, not a threshold calibrated against verified labels.
- **`name_scorer="fuzzy"`, `min_name_similarity=0.85`**: the package's RapidFuzz token-set name score must also be at least 0.85. Larger is stricter. A perfect token-set score is not proof of identity.
- **`neighbors=50`**: search up to 50 nearest other rows per record. With 4,000 rows this is **not exhaustive candidate enumeration**. Changing this cap can change linkage coverage; no candidate-recall guarantee is claimed. `max_rows_per_dataset` is left unset: no input-row sampling.

`result.records` contains all rows with nonblank names. `result.pairs` contains distance-qualified candidates, including those rejected by name scoring. `result.groups_frame` contains records connected by accepted pairs, without unmatched singletons. Matching can occur **within a source**, and clustering can form transitive multi-record groups. There is no cross-source-only or one-to-one constraint.

In [4]:
config = MatchConfig(
    max_distance=0.15, min_name_similarity=0.85, name_scorer="fuzzy", neighbors=50
)
result = await matcher.match(datasets, config)
assert result.records.height == valid_input_rows
assert result.records["record_id"].n_unique() == valid_input_rows
assert result.embeddings.shape[0] == valid_input_rows
assert np.isfinite(result.embeddings).all()
show(Markdown("**Unified records: first 5 rows**"))
show(result.records.head(5))
show(Markdown("**Sentences passed to the embedder: first 2**"))
for sentence in to_sentences(result.records)[:2]:
    print(sentence)

rows = result.records.to_dicts()
candidate_details = pl.DataFrame(
    [
        {
            "left_id": rows[pair.left]["record_id"],
            "right_id": rows[pair.right]["record_id"],
            "cross_source": rows[pair.left]["dataset"] != rows[pair.right]["dataset"],
            "cosine_distance": pair.distance,
            "name_similarity": pair.name_similarity,
            "accepted": pair.name_similarity >= config.min_name_similarity,
        }
        for pair in result.pairs
    ],
    schema={
        "left_id": pl.String,
        "right_id": pl.String,
        "cross_source": pl.Boolean,
        "cosine_distance": pl.Float64,
        "name_similarity": pl.Float64,
        "accepted": pl.Boolean,
    },
)
show(
    Markdown(
        f"**Candidate preview: first 20 of {candidate_details.height} distance-qualified pairs**"
    )
)
show(candidate_details.head(20))
show(Markdown("**All candidate counts by source relation and name-filter decision**"))
show(
    candidate_details
    .group_by("cross_source", "accepted")
    .len()
    .sort("cross_source", "accepted")
)
print("Embedding matrix shape:", result.embeddings.shape)

**Unified records: first 5 rows**

record_id,dataset,entity_id,name,address,city,country,email,owner,phone,sector
str,str,str,str,str,str,str,str,str,str,str
"""data_1:1""","""data_1""","""1""","""Adventure Awaits""","""1212 Journey Rd""","""Explore City""","""Traveland""","""info@adventureawaits.com""","""Taylor Journey""","""1111000999""","""Travel"""
"""data_1:2""","""data_1""","""2""","""Adventure Gear""","""1010 Trekking Trl""","""Adventure City""","""Exploreland""","""contact@adventuregear.retail""","""Morgan Trek""","""1555000115""","""Retail"""
"""data_1:3""","""data_1""","""3""","""Adventure Gear Co.""","""555 Adventure St""","""Outdoor City""","""ExploreLand""","""info@adventuregear.com""","""Explorer Jane""","""6655443322""","""Retail"""
"""data_1:4""","""data_1""","""4""","""Adventure Gear Company""","""555 Adventure Street""","""Outdoor City""","""ExploreLand""","""contact@adventuregear.com""","""Explorer Jane""","""6655443322""","""Retail"""
"""data_1:5""","""data_1""","""5""","""Adventure Outdoors""","""654 Adventure Ln""","""Outdoorsville""","""Explorica""","""info@adventureoutdoors.com""","""Alex Explorer""","""3344556677""","""Recreation"""


**Sentences passed to the embedder: first 2**

NAME: ADVENTURE AWAITS; ADDRESS: 1212 JOURNEY RD; CITY: EXPLORE CITY; COUNTRY: TRAVELAND; EMAIL: INFO@ADVENTUREAWAITS.COM; OWNER: TAYLOR JOURNEY; PHONE: 1111000999; SECTOR: TRAVEL
NAME: ADVENTURE GEAR; ADDRESS: 1010 TREKKING TRL; CITY: ADVENTURE CITY; COUNTRY: EXPLORELAND; EMAIL: CONTACT@ADVENTUREGEAR.RETAIL; OWNER: MORGAN TREK; PHONE: 1555000115; SECTOR: RETAIL


**Candidate preview: first 20 of 2345 distance-qualified pairs**

left_id,right_id,cross_source,cosine_distance,name_similarity,accepted
str,str,bool,f64,f64,bool
"""data_1:260""","""data_2:266""",true,-1.1921e-7,1.0,true
"""data_1:375""","""data_2:381""",true,-1.1921e-7,1.0,true
"""data_1:1736""","""data_2:1722""",true,-1.1921e-7,1.0,true
"""data_1:1877""","""data_2:1884""",true,-1.1921e-7,1.0,true
"""data_1:203""","""data_2:212""",true,0.0,1.0,true
"""data_1:1655""","""data_2:1636""",true,0.0,1.0,true
"""data_2:1134""","""data_2:1135""",false,0.0,0.866667,true
"""data_1:277""","""data_2:279""",true,5.9605e-8,1.0,true
"""data_1:370""","""data_2:378""",true,5.9605e-8,1.0,true


**All candidate counts by source relation and name-filter decision**

cross_source,accepted,len
bool,bool,u32
false,false,684
false,true,394
true,false,651
true,true,616


Embedding matrix shape: (4000, 5043)


## 5. Read matched groups and unmatched records

Every group is a **proposed** identity cluster, not a confirmed entity. `name_similarity` is the mean accepted-edge name score, not a probability or combined name/address confidence. We recover unmatched records by anti-joining all unified records against matched `record_id`s. Unmatched means no accepted group at these settings, **not** proven absence of a counterpart.

The source summary distinguishes records in any group from records in a group spanning both inputs. The group summary separates cross-source from source-only groups; cross-source groups may still contain several records from the same input. Table previews are capped, but counts use complete results.

In [5]:
matched = result.groups_frame
unmatched = result.records.join(
    matched.select("record_id"), on="record_id", how="anti"
).sort("record_id")
group_details = pl.DataFrame(
    [
        {
            "group_id": group.id,
            "group_size": group.size,
            "data_1_records": sum(
                rows[index]["dataset"] == "data_1" for index in group.rows
            ),
            "data_2_records": sum(
                rows[index]["dataset"] == "data_2" for index in group.rows
            ),
            "name_similarity": group.name_similarity,
        }
        for group in result.groups
    ],
    schema={
        "group_id": pl.UInt32,
        "group_size": pl.UInt32,
        "data_1_records": pl.UInt32,
        "data_2_records": pl.UInt32,
        "name_similarity": pl.Float64,
    },
).with_columns(
    ((pl.col("data_1_records") > 0) & (pl.col("data_2_records") > 0)).alias(
        "cross_source"
    )
)
cross_source_group_ids = group_details.filter(pl.col("cross_source"))[
    "group_id"
].to_list()
source_summary = pl.DataFrame([
    {
        "source": dataset.name,
        "input_rows": dataset.frame.height,
        "unified_rows": result.records.filter(pl.col("dataset") == dataset.name).height,
        "grouped_records": matched.filter(pl.col("dataset") == dataset.name).height,
        "in_cross_source_groups": matched.filter(
            (pl.col("dataset") == dataset.name)
            & pl.col("group_id").is_in(cross_source_group_ids)
        ).height,
        "unmatched_records": unmatched.filter(pl.col("dataset") == dataset.name).height,
    }
    for dataset in datasets
])
show(Markdown("**Per-source counts (all rows)**"))
show(source_summary)
show(Markdown(f"**Group preview: first 20 of {len(result.groups)} proposed groups**"))
show(group_details.head(20))
show(
    Markdown(
        f"**Matched-record preview: first 20 of {matched.height} grouped records**"
    )
)
show(
    matched.select(
        "group_id",
        "group_size",
        "name_similarity",
        "record_id",
        "name",
        "address",
        "city",
        "country",
    ).head(20)
)
show(Markdown(f"**Unmatched preview: first 20 of {unmatched.height} records**"))
show(unmatched.head(20))
show(
    Markdown(
        "**Largest proposed groups: inspect for possible transitive over-linking**"
    )
)
show(group_details.sort("group_size", descending=True).head(5))
largest_group = max(
    result.groups, key=lambda group: (group.size, -group.id), default=None
)
if largest_group is not None:
    show(
        Markdown(
            f"**Group {largest_group.id}: first 15 of {largest_group.size} records. A large cluster is not evidence that all members are the same company.**"
        )
    )
    show(
        matched
        .filter(pl.col("group_id") == largest_group.id)
        .select("record_id", "name", "address", "city", "phone")
        .head(15)
    )
print(
    f"Observed: {len(result.groups)} groups, {len(cross_source_group_ids)} cross-source groups, {len(result.groups) - len(cross_source_group_ids)} source-only groups, {matched.height} grouped records, {unmatched.height} unmatched records."
)

**Per-source counts (all rows)**

source,input_rows,unified_rows,grouped_records,in_cross_source_groups,unmatched_records
str,i64,i64,i64,i64,i64
"""data_1""",2000,2000,553,465,1447
"""data_2""",2000,2000,573,472,1427


**Group preview: first 20 of 380 proposed groups**

group_id,group_size,data_1_records,data_2_records,name_similarity,cross_source
u32,u32,u32,u32,f64,bool
0,2,1,1,1.0,true
1,2,1,1,1.0,true
2,2,1,1,1.0,true
3,3,2,1,1.0,true
4,2,1,1,1.0,true
5,2,1,1,1.0,true
6,3,2,1,1.0,true
7,4,2,2,1.0,true
8,2,1,1,1.0,true


**Matched-record preview: first 20 of 1126 grouped records**

group_id,group_size,name_similarity,record_id,name,address,city,country
u32,u32,f64,str,str,str,str,str
0,2,1.0,"""data_1:5""","""Adventure Outdoors""","""654 Adventure Ln""","""Outdoorsville""","""Explorica"""
0,2,1.0,"""data_2:4""","""Adventure Outdoors""","""404 Adventure Ln""","""Explore City""","""Adventuria"""
1,2,1.0,"""data_1:8""","""Adventure Tours""","""1414 Journey Ave""","""Explore City""","""Traveland"""
1,2,1.0,"""data_2:10""","""Adventure Tours Co.""","""322 Explore St""","""Adventure City""","""Traveland"""
2,2,1.0,"""data_1:18""","""AeroDynamics Corp""","""127 Skyway Blvd""","""AeroCity""","""Skyland"""
2,2,1.0,"""data_2:21""","""AeroDynamics Corp""","""103 Skyway Blvd.""","""AeroCity""","""Skyland"""
3,3,1.0,"""data_1:65""","""Agriculture Innovators""","""33 Farm Rd""","""Agro City""","""Farmland"""
3,3,1.0,"""data_1:66""","""Agriculture Innovators Inc.""","""369 Farm Rd""","""Agri City""","""Farmville"""
3,3,1.0,"""data_2:65""","""Agriculture Innovators""","""1010 Farm Ln""","""Agro City""","""Farmland"""


... (3 outputs left out)

group_id,group_size,data_1_records,data_2_records,name_similarity,cross_source
u32,u32,u32,u32,f64,bool
308,77,39,38,0.934303,true
264,23,9,14,0.953343,true
303,18,9,9,0.936439,true
320,17,9,8,0.929345,true
317,16,9,7,0.933111,true


**Group 308: first 15 of 77 records. A large cluster is not evidence that all members are the same company.**

record_id,name,address,city,phone
str,str,str,str,str
"""data_1:238""","""Build & Construct""","""159 Build Blvd""","""ConstructCity""","""556677889"""
"""data_1:243""","""Build It""","""600 Build Rd""","""Construct City""","""6789012345"""
"""data_1:252""","""BuildIt""","""700 Build Rd""","""Construct City""","""5566778899"""
"""data_1:253""","""BuildIt Construction""","""456 Build Ave""","""Construct City""","""2233445566"""
"""data_1:254""","""BuildIt Construktion""","""456 Build Avenue""","""Construct City""","""2233445566"""
"""data_1:256""","""BuildIt Innovations""","""802 Build Rd""","""Construct City""","""5566778899"""
"""data_1:257""","""BuildIt Partners""","""710 Build St""","""ConstructCity""","""5566778899"""
"""data_1:258""","""BuildIt Solutions""","""803 Build Rd""","""Construct City""","""5566778899"""
"""data_1:261""","""BuildLand Innovations""","""10003 Constructive Rd""","""ConstructCity""","""9012345678"""


Observed: 380 groups, 288 cross-source groups, 92 source-only groups, 1126 grouped records, 2874 unmatched records.


## 6. Check result integrity, not identity accuracy

These checks verify bookkeeping: each valid input record occurs exactly once in the unified data; matched and unmatched records form a disjoint complete partition; groups have at least two records; and every accepted edge meets both thresholds and is represented in a group. They **cannot** establish which proposed links describe the same real-world company. No manually fabricated expectations or historical outputs are used.

In [6]:
all_ids = set(result.records["record_id"].to_list())
matched_ids = set(matched["record_id"].to_list())
unmatched_ids = set(unmatched["record_id"].to_list())
assert matched_ids.isdisjoint(unmatched_ids)
assert matched_ids | unmatched_ids == all_ids
assert matched.height == len(matched_ids)
assert matched.height + unmatched.height == valid_input_rows
assert all(group.size >= 2 for group in result.groups)
group_by_row = {index: group.id for group in result.groups for index in group.rows}
assert len(group_by_row) == matched.height
assert all(pair.distance <= config.max_distance for pair in result.pairs)
for pair in result.pairs:
    if pair.name_similarity >= config.min_name_similarity:
        assert group_by_row[pair.left] == group_by_row[pair.right]
assert sum(source_summary["grouped_records"]) == matched.height
assert sum(source_summary["unmatched_records"]) == unmatched.height
print(
    f"PASS: all {valid_input_rows} valid input records accounted for; group and candidate bookkeeping is consistent. Identity accuracy is unmeasured."
)

PASS: all 4000 valid input records accounted for; group and candidate bookkeeping is consistent. Identity accuracy is unmeasured.


## 7. Change thresholds without re-embedding

`Matcher.regroup` reuses `result.embeddings`. The table reports **observed structural changes only**: candidate and accepted-edge counts, cross-source/source-only groups, grouped/unmatched records, largest group, and implied pair links. Implied links count all pairs in each connected group, including pairs without a direct accepted edge.

Tighter distance cutoffs can leave more records unmatched; looser cutoffs or disabling name filtering can connect larger groups through chains. More links are not necessarily more correct links. We cannot label these changes as false matches or missed true matches without verified labels. `neighbors=50` stays fixed in this comparison.

In [7]:
calls_before = embedder.calls
sensitivity = []
for label, distance, name_threshold in [
    ("strict", 0.05, 0.85),
    ("demo", 0.15, 0.85),
    ("loose", 0.30, 0.85),
    ("no name filter", 0.30, 0.0),
]:
    regrouped = matcher.regroup(
        result,
        MatchConfig(
            max_distance=distance,
            min_name_similarity=name_threshold,
            name_scorer="fuzzy",
            neighbors=50,
        ),
    )
    assert regrouped.embeddings is result.embeddings
    grouped_rows = sum(group.size for group in regrouped.groups)
    cross_groups = 0
    cross_links = 0
    within_links = 0
    for group in regrouped.groups:
        count_1 = sum(rows[index]["dataset"] == "data_1" for index in group.rows)
        count_2 = group.size - count_1
        cross_groups += int(count_1 > 0 and count_2 > 0)
        cross_links += count_1 * count_2
        within_links += count_1 * (count_1 - 1) // 2 + count_2 * (count_2 - 1) // 2
    sensitivity.append({
        "setting": label,
        "max_distance": distance,
        "min_name_similarity": name_threshold,
        "candidates": len(regrouped.pairs),
        "accepted_edges": sum(
            pair.name_similarity >= name_threshold for pair in regrouped.pairs
        ),
        "groups": len(regrouped.groups),
        "cross_source_groups": cross_groups,
        "source_only_groups": len(regrouped.groups) - cross_groups,
        "grouped_records": grouped_rows,
        "unmatched_records": valid_input_rows - grouped_rows,
        "largest_group": max((group.size for group in regrouped.groups), default=0),
        "implied_cross_source_links": cross_links,
        "implied_within_source_links": within_links,
    })
    if label == "demo":
        assert regrouped.groups == result.groups
        assert regrouped.pairs == result.pairs
assert embedder.calls == calls_before == 1
sensitivity_frame = pl.DataFrame(sensitivity)
show(
    sensitivity_frame.select(
        "setting",
        "max_distance",
        "min_name_similarity",
        "candidates",
        "accepted_edges",
        "groups",
    )
)
show(
    sensitivity_frame.select(
        "setting",
        "cross_source_groups",
        "source_only_groups",
        "grouped_records",
        "unmatched_records",
        "largest_group",
    )
)
show(
    sensitivity_frame.select(
        "setting", "implied_cross_source_links", "implied_within_source_links"
    )
)
print("All regroup runs reused the original vectors; embedding calls:", embedder.calls)

setting,max_distance,min_name_similarity,candidates,accepted_edges,groups
str,f64,f64,i64,i64,i64
"""strict""",0.05,0.85,153,144,133
"""demo""",0.15,0.85,2345,1010,380
"""loose""",0.3,0.85,51689,5320,498
"""no name filter""",0.3,0.0,51689,51689,58


setting,cross_source_groups,source_only_groups,grouped_records,unmatched_records,largest_group
str,i64,i64,i64,i64,i64
"""strict""",84,49,273,3727,3
"""demo""",288,92,1126,2874,77
"""loose""",414,84,2787,1213,723
"""no name filter""",49,9,3814,186,2147


setting,implied_cross_source_links,implied_within_source_links
str,i64,i64
"""strict""",90,57
"""demo""",2563,2309
"""loose""",137102,136398
"""no name filter""",1223195,1222144


All regroup runs reused the original vectors; embedding calls: 1


## Interpretation and limitations

Use the saved counts and record previews to explore the existing inputs, not to certify company identities. The complete `matched` and `unmatched` frames can be filtered interactively by source, record ID, or group ID. No inputs were copied, and no new datasets were generated.

- **Identity accuracy is unknown.** Historical outputs are not labels. Manually adjudicate representative pairs and whole groups, then calibrate on separate labeled data before measuring precision/recall or making decisions.
- Character trigrams capture spelling overlap, not semantics. Shared labels, sectors, email components, and phone values can inflate similarity. Missing or incorrect addresses, name aliases, transliterations, and branches sharing locations require additional care.
- Token-set name scoring may return 1.0 when one name's tokens are a subset of another. Union-find can transitively join records without a direct similar pair. This pipeline supports within-source and many-to-many groups; it does not enforce one-to-one cross-source matching.
- Unmatched means no accepted group at the current thresholds and neighbor cap, not proven uniqueness. Blank names are excluded by the package and explicitly counted in the input summary.
- The dense in-memory lexical matrix is a demo adapter. The package computes chunked all-pairs cosine similarities even with `neighbors=50`; runtime and memory grow with record count and vector dimension.
- `Matcher.evaluate` is optional LLM group assessment requiring a chat function. It is intentionally **not called** here. An LLM score would not establish ground truth either.

For neural local embeddings, install the extra from the **local** core directory (`uv pip install --python .venv-match-demo\Scripts\python.exe -e ".\packages\itk-core[local]" -e .\packages\itk-match`), then use `Matcher(embedder=LocalEmbedder())` with `from itk_core.local_embedder import LocalEmbedder`. This adds sentence-transformers/torch and may download model weights on first use. Recalibrate thresholds. For hosted embeddings/evaluation, see the package README's `Matcher.from_openai` example; hosted calls require credentials and may cost money.